# 기존 원판 모델이 박스를 내지 않을 때 CutMix로 보완하기

검증 6명의 원본 687프레임에서 기존 Mac 베이스라인의 선택을 우선합니다. 기존 출력이 비고 CutMix가 박스를 냈을 때만 그 박스를 씁니다. 두 모델은 각각 0번 프레임부터 자기 궤적을 유지했습니다. 저장된 결과를 재생하므로 두 모델을 서비스에서 함께 실행하는 시간은 이 실험에서 측정하지 않습니다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from wlcv.evaluation.plate_cutmix_fallback import replay, check_training_backgrounds

folder = root / "outputs/plate_cutmix_fallback"
frames, summary = replay()

## 사람이 그린 박스와 비교

채점한 270장에서는 수동 박스와 IoU 0.50 이상이면 정탐입니다. 아래 표는 기존 모델의 판정과 보완 후 판정을 같은 프레임에서 비교합니다.

In [ ]:
display(summary)
filled = frames[frames.used_cutmix & frames.method.eq("Past link")]
display(filled[["video_id", "frame_index", "in_motion", "source", "confidence", "iou", "match"]])

In [ ]:
totals = summary[summary["평가 구간"].eq("동작 중 전체")].set_index("방법")
data = totals.loc[["YOLO", "Past link"], ["기존 미탐", "비교 미탐"]].copy()
data.index = ["Direct", "Temporal link"]
data.columns = ["Mac baseline", "Baseline + CutMix fallback"]
ax = data.plot.bar(color=["#40546b", "#44817b"], rot=0, figsize=(7, 4))
ax.set(title="Misses in 270 scored motion frames", xlabel="", ylabel="Missed frames")
for bars in ax.containers:
    ax.bar_label(bars)
ax.figure.tight_layout()
ax.figure.savefig(folder / "misses.png", dpi=150)
plt.show()

## 학습 배경에서 오탐 확인

b05와 b06은 원판이 없는 학습 영상입니다. 두 영상의 총 176프레임을 다시 추론하려면 아래 값을 True로 바꾸세요. 새 배경에 대한 독립 검증은 별도입니다.

In [ ]:
recheck_background = False
if recheck_background:
    display(check_training_backgrounds())
elif (folder / "training_backgrounds.csv").is_file():
    display(pd.read_csv(folder / "training_backgrounds.csv"))